# 08 — Explainability Analysis
SHAP + attention maps + gate weight analysis.

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import json, warnings
import numpy as np
import pandas as pd
import torch
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import xgboost as xgb
from datasets import load_dataset
from transformers import DistilBertTokenizerFast

from src.fusion_model import HybridSalesPredictor
from src.explainability import HybridExplainer

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
MODELS_DIR  = ROOT / 'models'
FIGURES_DIR = ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)
DEVICE = 'cpu'
SAMPLE_SIZE = 2000
MAX_LEN = 128
print('Setup done.')

## Load data & models

In [ ]:
ds = load_dataset('DeepMostInnovations/saas-sales-conversations', split='train')
df = ds.to_pandas().sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
y  = df['outcome'].astype(int).values

num_cols = [c for c in ['conversation_length','customer_engagement','sales_effectiveness'] if c in df.columns]
cat_cols = [c for c in ['product_type','conversation_style','communication_channel'] if c in df.columns]
tab_df = df[num_cols + cat_cols].copy()
if 'customer_engagement' in tab_df.columns and 'sales_effectiveness' in tab_df.columns:
    tab_df['eng_eff_ratio'] = tab_df['customer_engagement'] / (tab_df['sales_effectiveness'] + 0.001)
fin_num = [c for c in tab_df.columns if c not in cat_cols]
scaler = StandardScaler()
X_num = scaler.fit_transform(tab_df[fin_num].fillna(0).values)
if cat_cols:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_cat = ohe.fit_transform(tab_df[cat_cols].fillna('unknown').astype(str))
    X_tab = np.hstack([X_num, X_cat])
    feature_names = fin_num + ohe.get_feature_names_out(cat_cols).tolist()
else:
    X_tab = X_num
    feature_names = fin_num

# Load XGBoost
xgb_path = MODELS_DIR / 'xgboost_model.json'
if xgb_path.exists():
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model(str(xgb_path))
    print('Loaded saved XGBoost model.')
else:
    print('Training XGBoost from scratch...')
    xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                    eval_metric='logloss', random_state=SEED, verbosity=0)
    xgb_model.fit(X_tab, y)

# Load hybrid model (if saved)
model_path = MODELS_DIR / 'hybrid_model.pt'
hybrid_model = None
if model_path.exists():
    hybrid_model = torch.load(str(model_path), map_location=DEVICE)
    hybrid_model.eval()
    print('Loaded hybrid model.')
else:
    print('hybrid_model.pt not found — run notebook 06 first. Skipping DL explainability.')

print(f'X_tab: {X_tab.shape}  Features: {len(feature_names)}')

## SHAP — Feature importance

In [ ]:
print('[1/4] SHAP summary plot...')
explainer = shap.TreeExplainer(xgb_model)
shap_vals = explainer.shap_values(X_tab[:500])  # sample 500 for speed

fig, ax = plt.subplots(figsize=(9, 6))
shap.summary_plot(shap_vals, X_tab[:500], feature_names=feature_names,
                   show=False, max_display=15)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()
print('  Saved shap_summary.png')

## SHAP waterfall for individual examples

In [ ]:
print('[2/4] SHAP waterfall for 3 individual samples...')
sample_indices = [0, 1, 2]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax_i, si in enumerate(sample_indices):
    sv = shap_vals[si]
    base = explainer.expected_value
    sorted_pairs = sorted(zip(feature_names, sv), key=lambda x: abs(x[1]), reverse=True)[:8]
    names_s  = [p[0] for p in sorted_pairs]
    values_s = [p[1] for p in sorted_pairs]
    colors = ['#e74c3c' if v > 0 else '#3498db' for v in values_s]
    axes[ax_i].barh(names_s, values_s, color=colors)
    axes[ax_i].axvline(0, color='black', linewidth=0.8)
    axes[ax_i].set_title(f'Sample {si} (y={y[si]}, pred={xgb_model.predict([X_tab[si]])[0]})')
    axes[ax_i].set_xlabel('SHAP Value')

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'shap_waterfall_samples.png'), dpi=150)
plt.show()
print('  Saved shap_waterfall_samples.png')

## Attention map (requires hybrid model)

In [ ]:
if hybrid_model is None:
    print('[3/4] Skipping attention map — hybrid model not loaded.')
else:
    print('[3/4] Attention heatmap for example conversations...')
    tok = DistilBertTokenizerFast.from_pretrained(
        str(MODELS_DIR / 'tokenizer') if (MODELS_DIR / 'tokenizer').exists() else 'distilbert-base-uncased'
    )
    text_col = 'full_text' if 'full_text' in df.columns else 'conversation'
    example_texts = df[text_col].fillna('').astype(str).head(3).tolist()

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax_i, text in enumerate(example_texts):
        enc = tok([text], max_length=MAX_LEN, padding='max_length',
                   truncation=True, return_tensors='pt')
        # Dummy leaf for attention extraction
        cfg = json.load(open(MODELS_DIR / 'feature_config.json')) if (MODELS_DIR / 'feature_config.json').exists() else {'leaf_dim': 100}
        leaf_dim = cfg.get('leaf_dim', 100)
        dummy_leaf = torch.zeros(1, leaf_dim)
        if hybrid_model.tab_projection is not None and list(hybrid_model.tab_projection.parameters())[0].shape[1] == leaf_dim:
            with torch.no_grad():
                _, _, attn_w = hybrid_model(dummy_leaf, enc['input_ids'], enc['attention_mask'])
            w = attn_w[0].cpu().numpy()
            axes[ax_i].bar(range(len(w)), w, color='coral', alpha=0.7)
            axes[ax_i].set_title(f'Conv {ax_i} (y={y[ax_i]})')
            axes[ax_i].set_xlabel('Token position')
            axes[ax_i].set_ylabel('Attention weight')
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / 'attention_heatmaps.png'), dpi=150)
    plt.show()
    print('  Saved attention_heatmaps.png')

## Gate weight analysis

In [ ]:
if hybrid_model is None:
    print('[4/4] Skipping gate analysis — hybrid model not loaded.')
else:
    print('[4/4] Gate weight distribution...')
    from torch.utils.data import DataLoader, TensorDataset

    tok = DistilBertTokenizerFast.from_pretrained(
        str(MODELS_DIR / 'tokenizer') if (MODELS_DIR / 'tokenizer').exists() else 'distilbert-base-uncased'
    )
    text_col = 'full_text' if 'full_text' in df.columns else 'conversation'
    texts = df[text_col].fillna('').astype(str).head(200).tolist()
    enc = tok(texts, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')

    cfg = json.load(open(MODELS_DIR / 'feature_config.json')) if (MODELS_DIR / 'feature_config.json').exists() else {'leaf_dim': 100}
    leaf_dim = cfg.get('leaf_dim', 100)
    dummy_leaves = torch.zeros(len(texts), leaf_dim)

    all_gate_means = []
    ds_g = TensorDataset(dummy_leaves, enc['input_ids'], enc['attention_mask'])
    dl_g = DataLoader(ds_g, batch_size=16)
    with torch.no_grad():
        for lb, ids, masks in dl_g:
            try:
                _, gate, _ = hybrid_model(lb, ids, masks)
                all_gate_means.extend(gate.mean(dim=1).numpy().tolist())
            except Exception:
                pass

    if all_gate_means:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(all_gate_means, bins=30, color='steelblue', edgecolor='white')
        ax.axvline(np.mean(all_gate_means), color='red', linestyle='--',
                   label=f'Mean={np.mean(all_gate_means):.3f}')
        ax.set_xlabel('Mean Gate Value (0=trust text, 1=trust tabular)')
        ax.set_ylabel('Count')
        ax.set_title('Gate Weight Distribution — Tabular vs Text Reliance')
        ax.legend()
        plt.tight_layout()
        plt.savefig(str(FIGURES_DIR / 'gate_distribution.png'), dpi=150)
        plt.show()
        print(f'  Mean gate (tabular reliance): {np.mean(all_gate_means):.3f}')
        print('  Saved gate_distribution.png')
    else:
        print('  Could not extract gate values (leaf_dim mismatch — run notebook 06 first).')

print('\nDone ✅')